In [1]:
import numpy as np
import pandas as pd

In [2]:
data_claim = pd.read_csv('data/Data_Klaim.csv')
data_polis = pd.read_csv('data/Data_Polis.csv')

ARIMA

In [3]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. RAW DATA TO WEEKLY AGGREGATION
# ==========================================
print("1. Processing Raw Data to Weekly Level...")

data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

# Resample to Weekly (W-SUN)
weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()

weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)

# Calculate Severity
weekly_df['Severity'] = np.where(weekly_df['Frequency'] > 0, 
                                 weekly_df['Total_Claim'] / weekly_df['Frequency'], 0)

# ==========================================
# 2. PURE ARIMA TRAINING
# ==========================================
print("2. Training Pure ARIMA(1,1,1) Models...")

# We train directly on the sequence of numbers (ignoring the dates entirely for the math)
model_freq = ARIMA(weekly_df['Frequency'], order=(1, 1, 1)).fit()
model_sev = ARIMA(weekly_df['Severity'], order=(1, 1, 1)).fit()

print("\n[FREQUENCY MODEL SUMMARY]")
print(model_freq.summary().tables[1])

# ==========================================
# 3. FORECASTING FUTURE WEEKS
# ==========================================
print("\n3. Forecasting Future Weeks...")

last_hist_date = weekly_df['Week_End_Date'].max()
target_end_date = pd.to_datetime('2025-12-31')

forecast_weeks = pd.date_range(start=last_hist_date + pd.Timedelta(days=7), 
                               end=target_end_date + pd.Timedelta(days=7), freq='W-SUN')

steps_to_forecast = len(forecast_weeks)

# Pure ARIMA simply projects forward N steps based on momentum and trend
future_freq = model_freq.forecast(steps=steps_to_forecast).values
future_sev = model_sev.forecast(steps=steps_to_forecast).values

weekly_predictions = []

for i, week_end in enumerate(forecast_weeks):
    # Ensure no negative predictions (ARIMA can sometimes draw a line into the negatives)
    pred_freq = max(0, future_freq[i])
    pred_sev = max(0, future_sev[i])
    pred_total = pred_freq * pred_sev
    
    weekly_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6),
        'Week_End_Date': week_end,
        'Frequency': pred_freq,
        'Total_Claim': pred_total
    })

# ==========================================
# 4. DAILY APPORTIONMENT & MONTHLY ROLL-UP
# ==========================================
print("4. Apportioning to Daily and Rolling up to Exact Calendar Months...")

daily_records = []
for row in weekly_predictions:
    days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
    daily_freq = row['Frequency'] / 7.0
    daily_total = row['Total_Claim'] / 7.0
    for d in days:
        daily_records.append({'Date': d, 'Daily_Freq': daily_freq, 'Daily_Total': daily_total})

daily_df = pd.DataFrame(daily_records)
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

# The Pandas Date-Summing Crash Fix (Explicitly selecting numeric columns)
cols_to_sum = ['Daily_Freq', 'Daily_Total']
monthly_forecast = daily_df.groupby('Month_Period')[cols_to_sum].sum().reset_index()

# Filter ONLY for Target Window
target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
final_forecast = monthly_forecast[(monthly_forecast['Month_Period'] >= target_start) & 
                                  (monthly_forecast['Month_Period'] <= target_end)].copy()

# The Mathematical Guarantee Rule
final_forecast['Frequency'] = np.round(final_forecast['Daily_Freq']).astype(int)
final_forecast['Total_Claim'] = final_forecast['Daily_Total']
final_forecast['Severity'] = final_forecast['Total_Claim'] / final_forecast['Frequency']

# ==========================================
# 5. EXPORT
# ==========================================
formatted_data = []
for index, row in final_forecast.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_pure_arima.csv', index=False)

print("\n--- FINAL FORECAST (Pure ARIMA(1,1,1)) ---")
print(final_forecast[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nFile 'submission_pure_arima.csv' saved!")

1. Processing Raw Data to Weekly Level...
2. Training Pure ARIMA(1,1,1) Models...

[FREQUENCY MODEL SUMMARY]
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.1570      0.115      1.368      0.171      -0.068       0.382
ma.L1         -0.9991      1.468     -0.680      0.496      -3.877       1.879
sigma2       150.9164    216.494      0.697      0.486    -273.405     575.237

3. Forecasting Future Weeks...
4. Apportioning to Daily and Rolling up to Exact Calendar Months...

--- FINAL FORECAST (Pure ARIMA(1,1,1)) ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        220  5.410827e+07  1.190382e+10
1      2025-09        239  5.371854e+07  1.283873e+10
2      2025-10        247  5.371144e+07  1.326673e+10
3      2025-11        239  5.371869e+07  1.283877e+10
4      2025-12        247  5.371144e+07  1.326673e+10

File 'submission_pure_arima

Prophet

In [4]:
import pandas as pd
import numpy as np
from prophet import Prophet
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. RAW DATA TO WEEKLY AGGREGATION
# ==========================================
print("1. Processing Raw Data to Weekly Level...")

data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

# The legendary Weekly Aggregation
weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()

weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)
weekly_df['Severity'] = np.where(weekly_df['Frequency'] > 0, 
                                 weekly_df['Total_Claim'] / weekly_df['Frequency'], 0)

# ==========================================
# 2. PROPHET TRAINING
# ==========================================
print("2. Formatting Data and Training Prophet Models...")

# Prophet absolutely requires the columns to be named 'ds' (datestamp) and 'y' (target)
df_prophet_freq = pd.DataFrame({
    'ds': weekly_df['Week_End_Date'],
    'y': weekly_df['Frequency']
})

df_prophet_sev = pd.DataFrame({
    'ds': weekly_df['Week_End_Date'],
    'y': weekly_df['Severity']
})

# Initialize and fit the models
# We leave parameters default, Prophet is incredibly smart out of the box
model_freq = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
model_freq.fit(df_prophet_freq)

model_sev = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
model_sev.fit(df_prophet_sev)

# ==========================================
# 3. FORECASTING FUTURE WEEKS
# ==========================================
print("3. Asking Prophet to generate the future...")

last_hist_date = weekly_df['Week_End_Date'].max()
target_end_date = pd.to_datetime('2025-12-31')

forecast_weeks = pd.date_range(start=last_hist_date + pd.Timedelta(days=7), 
                               end=target_end_date + pd.Timedelta(days=7), freq='W-SUN')

steps_to_forecast = len(forecast_weeks)

# Prophet uses a special dataframe builder for future dates
future_dataframe = model_freq.make_future_dataframe(periods=steps_to_forecast, freq='W-SUN')

# Generate the predictions (Prophet gives us a huge dataframe back, we just want 'yhat')
forecast_freq_full = model_freq.predict(future_dataframe)
forecast_sev_full = model_sev.predict(future_dataframe)

# Extract just the future predictions (ignoring the historical in-sample predictions)
future_freq_preds = forecast_freq_full['yhat'].tail(steps_to_forecast).values
future_sev_preds = forecast_sev_full['yhat'].tail(steps_to_forecast).values

weekly_predictions = []

for i, week_end in enumerate(forecast_weeks):
    # Ensure no negative predictions (Prophet's trendlines can dip below zero)
    pred_freq = max(0, future_freq_preds[i])
    pred_sev = max(0, future_sev_preds[i])
    pred_total = pred_freq * pred_sev
    
    weekly_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6),
        'Week_End_Date': week_end,
        'Frequency': pred_freq,
        'Total_Claim': pred_total
    })

# ==========================================
# 4. DAILY APPORTIONMENT & MONTHLY ROLL-UP
# ==========================================
print("4. Apportioning to Daily and Rolling up to Exact Calendar Months...")

daily_records = []
for row in weekly_predictions:
    days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
    daily_freq = row['Frequency'] / 7.0
    daily_total = row['Total_Claim'] / 7.0
    for d in days:
        daily_records.append({'Date': d, 'Daily_Freq': daily_freq, 'Daily_Total': daily_total})

daily_df = pd.DataFrame(daily_records)
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

cols_to_sum = ['Daily_Freq', 'Daily_Total']
monthly_forecast = daily_df.groupby('Month_Period')[cols_to_sum].sum().reset_index()

target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
final_forecast = monthly_forecast[(monthly_forecast['Month_Period'] >= target_start) & 
                                  (monthly_forecast['Month_Period'] <= target_end)].copy()

final_forecast['Frequency'] = np.round(final_forecast['Daily_Freq']).astype(int)
final_forecast['Total_Claim'] = final_forecast['Daily_Total']
final_forecast['Severity'] = final_forecast['Total_Claim'] / final_forecast['Frequency']

# ==========================================
# 5. EXPORT
# ==========================================
formatted_data = []
for index, row in final_forecast.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_prophet.csv', index=False)

print("\n--- FINAL FORECAST (Facebook Prophet) ---")
print(final_forecast[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nFile 'submission_prophet.csv' saved!")

1. Processing Raw Data to Weekly Level...
2. Formatting Data and Training Prophet Models...


10:27:38 - cmdstanpy - INFO - Chain [1] start processing
10:27:38 - cmdstanpy - INFO - Chain [1] done processing
10:27:38 - cmdstanpy - INFO - Chain [1] start processing
10:27:38 - cmdstanpy - INFO - Chain [1] done processing


3. Asking Prophet to generate the future...
4. Apportioning to Daily and Rolling up to Exact Calendar Months...

--- FINAL FORECAST (Facebook Prophet) ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        210  5.342816e+07  1.121991e+10
1      2025-09        223  5.350270e+07  1.193110e+10
2      2025-10        229  5.341514e+07  1.223207e+10
3      2025-11        220  5.338223e+07  1.174409e+10
4      2025-12        226  5.326380e+07  1.203762e+10

File 'submission_prophet.csv' saved!


N-BEATS

In [5]:
import pandas as pd
import numpy as np
import warnings
import logging
warnings.filterwarnings('ignore')
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING) # Suppresses PyTorch console spam

from darts import TimeSeries
from darts.models import NBEATSModel
from darts.dataprocessing.transformers import Scaler

# ==========================================
# 1. RAW DATA TO WEEKLY AGGREGATION
# ==========================================
print("1. Processing Raw Data to Weekly Level...")

data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

# The Weekly Apportionment Baseline
weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()

weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)
weekly_df['Severity'] = np.where(weekly_df['Frequency'] > 0, 
                                 weekly_df['Total_Claim'] / weekly_df['Frequency'], 0)

# ==========================================
# 2. DARTS FORMATTING & SCALING
# ==========================================
print("2. Formatting for Darts and Scaling Data...")
# Deep Learning networks explode if they look at massive raw numbers (like millions of Rupiah).
# We MUST scale the data down to a 0-to-1 range before feeding it to N-BEATS.

series_freq = TimeSeries.from_dataframe(weekly_df, 'Week_End_Date', 'Frequency')
series_sev = TimeSeries.from_dataframe(weekly_df, 'Week_End_Date', 'Severity')

scaler_freq = Scaler()
scaler_sev = Scaler()

scaled_freq = scaler_freq.fit_transform(series_freq)
scaled_sev = scaler_sev.fit_transform(series_sev)

# ==========================================
# 3. N-BEATS TRAINING
# ==========================================
print("3. Compiling and Training N-BEATS Neural Networks...")

# input_chunk_length = 8 (Looks back at the last 2 months to find the pattern)
# output_chunk_length = 4 (Predicts 1 month ahead internally to learn the trajectory)
nbeats_params = {
    'input_chunk_length': 8,
    'output_chunk_length': 4,
    'n_epochs': 100,       # How many times it loops over your data to learn
    'random_state': 42,
    'pl_trainer_kwargs': {"accelerator": "auto"} # Auto-detects CPU/GPU
}

model_freq = NBEATSModel(**nbeats_params)
model_freq.fit(scaled_freq, verbose=False)

model_sev = NBEATSModel(**nbeats_params)
model_sev.fit(scaled_sev, verbose=False)

# ==========================================
# 4. FORECASTING & INVERSE SCALING
# ==========================================
print("4. Forecasting Future Weeks...")

last_hist_date = weekly_df['Week_End_Date'].max()
target_end_date = pd.to_datetime('2025-12-31')

forecast_weeks = pd.date_range(start=last_hist_date + pd.Timedelta(days=7), 
                               end=target_end_date + pd.Timedelta(days=7), freq='W-SUN')

steps_to_forecast = len(forecast_weeks)

# Predict the scaled values
pred_scaled_freq = model_freq.predict(n=steps_to_forecast)
pred_scaled_sev = model_sev.predict(n=steps_to_forecast)

# Inverse transform back to real-world numbers (raw claims and raw severity)
future_freq_preds = scaler_freq.inverse_transform(pred_scaled_freq).values().flatten()
future_sev_preds = scaler_sev.inverse_transform(pred_scaled_sev).values().flatten()

weekly_predictions = []

for i, week_end in enumerate(forecast_weeks):
    pred_freq = max(0, future_freq_preds[i])
    pred_sev = max(0, future_sev_preds[i])
    pred_total = pred_freq * pred_sev
    
    weekly_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6),
        'Week_End_Date': week_end,
        'Frequency': pred_freq,
        'Total_Claim': pred_total
    })

# ==========================================
# 5. DAILY APPORTIONMENT & MONTHLY ROLL-UP
# ==========================================
print("5. Apportioning to Daily and Rolling up to Exact Calendar Months...")

daily_records = []
for row in weekly_predictions:
    days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
    daily_freq = row['Frequency'] / 7.0
    daily_total = row['Total_Claim'] / 7.0
    for d in days:
        daily_records.append({'Date': d, 'Daily_Freq': daily_freq, 'Daily_Total': daily_total})

daily_df = pd.DataFrame(daily_records)
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

cols_to_sum = ['Daily_Freq', 'Daily_Total']
monthly_forecast = daily_df.groupby('Month_Period')[cols_to_sum].sum().reset_index()

target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
final_forecast = monthly_forecast[(monthly_forecast['Month_Period'] >= target_start) & 
                                  (monthly_forecast['Month_Period'] <= target_end)].copy()

final_forecast['Frequency'] = np.round(final_forecast['Daily_Freq']).astype(int)
final_forecast['Total_Claim'] = final_forecast['Daily_Total']
final_forecast['Severity'] = final_forecast['Total_Claim'] / final_forecast['Frequency']

# ==========================================
# 6. EXPORT
# ==========================================
formatted_data = []
for index, row in final_forecast.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_nbeats.csv', index=False)

print("\n--- FINAL FORECAST (N-BEATS Deep Learning) ---")
print(final_forecast[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nFile 'submission_nbeats.csv' saved!")

The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


1. Processing Raw Data to Weekly Level...
2. Formatting for Darts and Scaling Data...
3. Compiling and Training N-BEATS Neural Networks...


`Trainer.fit` stopped: `max_epochs=100` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

N-HiTS

In [ ]:
import pandas as pd
import numpy as np
import warnings
import logging
warnings.filterwarnings('ignore')
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING) # Suppresses PyTorch console spam

from darts import TimeSeries
from darts.models import NHiTSModel
from darts.dataprocessing.transformers import Scaler

# ==========================================
# 1. RAW DATA TO WEEKLY AGGREGATION
# ==========================================
print("1. Processing Raw Data to Weekly Level...")

data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

# The Weekly Apportionment Baseline
weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()

weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)
weekly_df['Severity'] = np.where(weekly_df['Frequency'] > 0, 
                                 weekly_df['Total_Claim'] / weekly_df['Frequency'], 0)

# ==========================================
# 2. DARTS FORMATTING & SCALING
# ==========================================
print("2. Formatting for Darts and Scaling Data...")
# Deep Learning networks explode if they look at massive raw numbers.
# We MUST scale the data down to a 0-to-1 range before feeding it to N-HiTS.

series_freq = TimeSeries.from_dataframe(weekly_df, 'Week_End_Date', 'Frequency')
series_sev = TimeSeries.from_dataframe(weekly_df, 'Week_End_Date', 'Severity')

scaler_freq = Scaler()
scaler_sev = Scaler()

scaled_freq = scaler_freq.fit_transform(series_freq)
scaled_sev = scaler_sev.fit_transform(series_sev)

# ==========================================
# 3. N-HITS TRAINING
# ==========================================
print("3. Compiling and Training N-HiTS Neural Networks...")

# input_chunk_length = 8 (Looks back at the last 2 months to find the pattern)
# output_chunk_length = 4 (Predicts 1 month ahead internally to learn the trajectory)
nhits_params = {
    'input_chunk_length': 8,
    'output_chunk_length': 4,
    'n_epochs': 100,       # How many times it loops over your data to learn
    'random_state': 42,
    'pl_trainer_kwargs': {"accelerator": "auto"} # Auto-detects CPU/GPU
}

# The ONLY thing that changed: We are using NHiTSModel now!
model_freq = NHiTSModel(**nhits_params)
model_freq.fit(scaled_freq, verbose=False)

model_sev = NHiTSModel(**nhits_params)
model_sev.fit(scaled_sev, verbose=False)

# ==========================================
# 4. FORECASTING & INVERSE SCALING
# ==========================================
print("4. Forecasting Future Weeks...")

last_hist_date = weekly_df['Week_End_Date'].max()
target_end_date = pd.to_datetime('2025-12-31')

forecast_weeks = pd.date_range(start=last_hist_date + pd.Timedelta(days=7), 
                               end=target_end_date + pd.Timedelta(days=7), freq='W-SUN')

steps_to_forecast = len(forecast_weeks)

# Predict the scaled values
pred_scaled_freq = model_freq.predict(n=steps_to_forecast)
pred_scaled_sev = model_sev.predict(n=steps_to_forecast)

# Inverse transform back to real-world numbers (raw claims and raw severity)
future_freq_preds = scaler_freq.inverse_transform(pred_scaled_freq).values().flatten()
future_sev_preds = scaler_sev.inverse_transform(pred_scaled_sev).values().flatten()

weekly_predictions = []

for i, week_end in enumerate(forecast_weeks):
    pred_freq = max(0, future_freq_preds[i])
    pred_sev = max(0, future_sev_preds[i])
    pred_total = pred_freq * pred_sev
    
    weekly_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6),
        'Week_End_Date': week_end,
        'Frequency': pred_freq,
        'Total_Claim': pred_total
    })

# ==========================================
# 5. DAILY APPORTIONMENT & MONTHLY ROLL-UP
# ==========================================
print("5. Apportioning to Daily and Rolling up to Exact Calendar Months...")

daily_records = []
for row in weekly_predictions:
    days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
    daily_freq = row['Frequency'] / 7.0
    daily_total = row['Total_Claim'] / 7.0
    for d in days:
        daily_records.append({'Date': d, 'Daily_Freq': daily_freq, 'Daily_Total': daily_total})

daily_df = pd.DataFrame(daily_records)
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

cols_to_sum = ['Daily_Freq', 'Daily_Total']
monthly_forecast = daily_df.groupby('Month_Period')[cols_to_sum].sum().reset_index()

target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
final_forecast = monthly_forecast[(monthly_forecast['Month_Period'] >= target_start) & 
                                  (monthly_forecast['Month_Period'] <= target_end)].copy()

final_forecast['Frequency'] = np.round(final_forecast['Daily_Freq']).astype(int)
final_forecast['Total_Claim'] = final_forecast['Daily_Total']
final_forecast['Severity'] = final_forecast['Total_Claim'] / final_forecast['Frequency']

# ==========================================
# 6. EXPORT
# ==========================================
formatted_data = []
for index, row in final_forecast.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_nhits.csv', index=False)

print("\n--- FINAL FORECAST (N-HiTS Deep Learning) ---")
print(final_forecast[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nFile 'submission_nhits.csv' saved!")

1. Processing Raw Data to Weekly Level...
2. Formatting for Darts and Scaling Data...
3. Compiling and Training N-HiTS Neural Networks...
4. Forecasting Future Weeks...


5. Apportioning to Daily and Rolling up to Exact Calendar Months...

--- FINAL FORECAST (N-HiTS Deep Learning) ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        232  4.446004e+07  1.031473e+10
1      2025-09        261  6.731341e+07  1.756880e+10
2      2025-10        262  4.792904e+07  1.255741e+10
3      2025-11        261  4.889726e+07  1.276218e+10
4      2025-12        265  6.268433e+07  1.661135e+10

File 'submission_nhits.csv' saved!


In [6]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. RAW DATA TO WEEKLY AGGREGATION
# ==========================================
print("1. Processing Raw Data to Weekly Level...")

# Load raw data (Adjust path if needed)
data_claim = pd.read_csv('data/Data_Klaim.csv')

# Clean and filter for valid claims
data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

# Resample to Weekly (Sunday)
weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()

weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)

# ==========================================
# 2. DEFINE FORECASTING HORIZON
# ==========================================
print("2. Setting up the Forecasting Timeline...")

last_hist_date = weekly_df['Week_End_Date'].max()
target_end_date = pd.to_datetime('2025-12-31')

# Generate the exact sequence of Sundays we need to predict
forecast_weeks = pd.date_range(
    start=last_hist_date + pd.Timedelta(days=7), 
    end=target_end_date + pd.Timedelta(days=7), 
    freq='W-SUN'
)

# ==========================================
# 3. GENERATE BASELINE PREDICTIONS
# ==========================================
print("3. Generating Naive and Seasonal Naive Forecasts...")

# --- A. PURE NAIVE FORECAST ---
# Logic: "The future will be exactly the same as the very last known week."
last_known_freq = weekly_df['Frequency'].iloc[-1]
last_known_tot = weekly_df['Total_Claim'].iloc[-1]

naive_predictions = []
for week_end in forecast_weeks:
    naive_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6),
        'Week_End_Date': week_end,
        'Frequency': last_known_freq,
        'Total_Claim': last_known_tot
    })

# --- B. SEASONAL NAIVE FORECAST ---
# Logic: "This week will be exactly the same as this exact week 1 year ago."
snaive_predictions = []
for week_end in forecast_weeks:
    target_past_date = week_end - pd.Timedelta(weeks=52)
    past_record = weekly_df[weekly_df['Week_End_Date'] == target_past_date]
    
    if not past_record.empty:
        s_freq = past_record['Frequency'].values[0]
        s_tot = past_record['Total_Claim'].values[0]
    else:
        # Fallback to Pure Naive if we don't have data going exactly 52 weeks back
        s_freq = last_known_freq
        s_tot = last_known_tot
        
    snaive_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6),
        'Week_End_Date': week_end,
        'Frequency': s_freq,
        'Total_Claim': s_tot
    })

# ==========================================
# 4. DAILY APPORTIONMENT & MONTHLY ROLL-UP FUNCTION
# ==========================================
print("4. Apportioning to Daily and Rolling up to Exact Calendar Months...")

def process_to_monthly_submission(predictions_list, output_filename):
    """Takes weekly predictions, splits them into days, and groups by exact Kaggle months."""
    daily_records = []
    
    # Split weekly into daily
    for row in predictions_list:
        days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
        daily_freq = row['Frequency'] / 7.0
        daily_total = row['Total_Claim'] / 7.0
        for d in days:
            daily_records.append({'Date': d, 'Daily_Freq': daily_freq, 'Daily_Total': daily_total})

    daily_df = pd.DataFrame(daily_records)
    daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

    # Group by Exact Month
    monthly_forecast = daily_df.groupby('Month_Period')[['Daily_Freq', 'Daily_Total']].sum().reset_index()

    # Filter for the exact Kaggle target window (August 2025 to December 2025)
    target_start = pd.Period('2025-08', freq='M')
    target_end = pd.Period('2025-12', freq='M')
    final_forecast = monthly_forecast[(monthly_forecast['Month_Period'] >= target_start) & 
                                      (monthly_forecast['Month_Period'] <= target_end)].copy()

    # Calculate final submission columns
    final_forecast['Frequency'] = np.round(final_forecast['Daily_Freq']).astype(int)
    final_forecast['Total_Claim'] = final_forecast['Daily_Total']
    
    # Calculate Severity (Prevent division by zero)
    final_forecast['Severity'] = np.where(final_forecast['Frequency'] > 0, 
                                          final_forecast['Total_Claim'] / final_forecast['Frequency'], 
                                          0)

    # Format for Kaggle Submission
    formatted_data = []
    for index, row in final_forecast.iterrows():
        m_id = str(row['Month_Period']).replace('-', '_')
        formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
        formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
        formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

    submission_df = pd.DataFrame(formatted_data)
    submission_df.to_csv(output_filename, index=False)
    
    print(f"\n--- Output Saved: {output_filename} ---")
    print(final_forecast[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])

# ==========================================
# 5. EXECUTE EXPORT
# ==========================================
process_to_monthly_submission(naive_predictions, 'submission_naive.csv')
process_to_monthly_submission(snaive_predictions, 'submission_snaive.csv')

print("\nProcess Complete! You now have your Level 0 Baselines.")

1. Processing Raw Data to Weekly Level...
2. Setting up the Forecasting Timeline...
3. Generating Naive and Seasonal Naive Forecasts...
4. Apportioning to Daily and Rolling up to Exact Calendar Months...

--- Output Saved: submission_naive.csv ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        160  4.329779e+07  6.927647e+09
1      2025-09        171  4.340631e+07  7.422479e+09
2      2025-10        177  4.333274e+07  7.669895e+09
3      2025-11        171  4.340631e+07  7.422479e+09
4      2025-12        177  4.333274e+07  7.669895e+09

--- Output Saved: submission_snaive.csv ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        209  6.003495e+07  1.254730e+10
1      2025-09        208  5.669271e+07  1.179208e+10
2      2025-10        275  4.645127e+07  1.277410e+10
3      2025-11        268  5.199788e+07  1.393543e+10
4      2025-12        238  4.966898e+07  1.182122e+10

Process Complete! You now have your Level 0 Baselines.
